# 12 — Siamese Vision Transformer on FCGR k=6

## Obiettivo

Valutare se un Vision Transformer puro riesca ad apprendere una
rappresentazione delle FCGR più discriminativa rispetto alla CNN V3.

Confronto controllato:

- FCGR k=6, 64×64
- stesse 12 classi Tumor + Healthy
- stesso downsampling a 2111 campioni/classe
- stesse random pairs 50/50
- stesso embedding 128D
- stessa Euclidean Contrastive Loss
- stesso margin = 1.25

Viene modificato esclusivamente l'encoder:

CNN V3 → Vision Transformer.

Configurazione ViT:

- image size: 64×64
- patch size: 8×8
- 64 patch tokens
- embedding dimension: 128
- 4 attention heads
- 4 Transformer blocks
- MLP dimension: 512
- CLS token apprendibile
- positional embedding apprendibile

In [1]:
# ============================================================
# IMPORTS + CONFIG
# ============================================================

from pathlib import Path

import random
import time
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score


# ============================================================
# PATHS
# ============================================================

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

FCGR_DIR = (
    PROCESSED_DIR
    / "fcgr_cache"
)

ARTIFACTS_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "siamese_vit_k6"
)

ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


MANIFEST_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_manifest.tsv"
)

CLASS_MAPPING_PATH = (
    PROCESSED_DIR
    / "siamese_tumor_healthy_class_mapping.tsv"
)

VAL_POOL_PATH = (
    PROCESSED_DIR
    / "siamese_val_pair_pool.tsv"
)

FCGR_PATH = (
    FCGR_DIR
    / "fcgr_k6.npy"
)

FCGR_INDEX_PATH = (
    FCGR_DIR
    / "fcgr_k6_index.tsv"
)


# ============================================================
# DATA
# ============================================================

K = 6

IMAGE_SIZE = 64

N_CLASSES = 12

RANDOM_STATE = 42


# ============================================================
# SIAMESE
# ============================================================

EMBEDDING_DIM = 128

EUCLIDEAN_MARGIN = 1.25

TRAIN_PAIRS_PER_EPOCH = 50_000

VAL_PAIRS = 10_000

POSITIVE_FRACTION = 0.50


# ============================================================
# VIT
# ============================================================

PATCH_SIZE = 8

VIT_DIM = 128

VIT_HEADS = 4

VIT_LAYERS = 4

VIT_MLP_DIM = 512

VIT_DROPOUT = 0.10


# ============================================================
# TRAINING
# ============================================================

BATCH_SIZE = 64

LEARNING_RATE = 3e-4

WEIGHT_DECAY = 1e-4


# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

AMP_ENABLED = (
    DEVICE.type == "cuda"
)


print("=" * 72)
print("SIAMESE ViT k=6")
print("=" * 72)

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print()
print("Image size:", IMAGE_SIZE)
print("Patch size:", PATCH_SIZE)
print(
    "Number patches:",
    (IMAGE_SIZE // PATCH_SIZE) ** 2
)
print("ViT dim:", VIT_DIM)
print("Heads:", VIT_HEADS)
print("Layers:", VIT_LAYERS)
print("Embedding:", EMBEDDING_DIM)

SIAMESE ViT k=6
Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU

Image size: 64
Patch size: 8
Number patches: 64
ViT dim: 128
Heads: 4
Layers: 4
Embedding: 128


In [2]:
# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(
    RANDOM_STATE
)


if DEVICE.type == "cuda":

    torch.backends.cudnn.benchmark = True

    torch.set_float32_matmul_precision(
        "high"
    )

In [3]:
# ============================================================
# LOAD MANIFEST
# ============================================================

metadata = pd.read_csv(
    MANIFEST_PATH,
    sep="\t",
    dtype={"id": str}
)

metadata["class_id"] = (
    metadata["class_id"]
    .astype(int)
)


class_mapping = pd.read_csv(
    CLASS_MAPPING_PATH,
    sep="\t"
)


# ============================================================
# TRAIN
# ============================================================

full_train_metadata = (
    metadata[
        metadata["split_cluster"]
        ==
        "train"
    ]
    .copy()
    .reset_index(drop=True)
)


train_counts = (
    full_train_metadata["class_id"]
    .value_counts()
    .sort_index()
)


MIN_CLASS_SIZE = int(
    train_counts.min()
)


balanced_parts = []


for class_id in sorted(
    full_train_metadata["class_id"].unique()
):

    class_df = (
        full_train_metadata[
            full_train_metadata["class_id"]
            ==
            class_id
        ]
    )


    sampled = class_df.sample(
        n=MIN_CLASS_SIZE,
        replace=False,
        random_state=(
            RANDOM_STATE
            +
            int(class_id)
        )
    )


    balanced_parts.append(
        sampled
    )


train_metadata = (
    pd.concat(
        balanced_parts,
        ignore_index=True
    )
    .reset_index(drop=True)
)


print("=" * 72)
print("TRAIN")
print("=" * 72)

print(
    "Full train:",
    len(full_train_metadata)
)

print(
    "Samples/class:",
    MIN_CLASS_SIZE
)

print(
    "Balanced:",
    len(train_metadata)
)

print(
    "Classes:",
    train_metadata["class_id"]
    .nunique()
)

TRAIN
Full train: 96167
Samples/class: 2111
Balanced: 25332
Classes: 12


In [4]:
# ============================================================
# VALIDATION
# ============================================================

old_to_new = dict(
    zip(
        class_mapping[
            "original_class_id"
        ].astype(int),

        class_mapping[
            "class_id"
        ].astype(int)
    )
)


included_original_ids = set(
    old_to_new.keys()
)


val_original = pd.read_csv(
    VAL_POOL_PATH,
    sep="\t",
    dtype={"id": str}
)


val_original["class_id"] = (
    val_original["class_id"]
    .astype(int)
)


val_metadata = (
    val_original[
        val_original["class_id"]
        .isin(
            included_original_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)


val_metadata[
    "original_class_id"
] = val_metadata[
    "class_id"
]


val_metadata[
    "class_id"
] = (
    val_metadata[
        "original_class_id"
    ]
    .map(old_to_new)
    .astype(int)
)


print(
    "Validation samples:",
    len(val_metadata)
)

print(
    "Validation classes:",
    val_metadata["class_id"]
    .nunique()
)

Validation samples: 9753
Validation classes: 12


In [5]:
# ============================================================
# LOAD FCGR k=6
# ============================================================

fcgr_memmap = np.load(
    FCGR_PATH,
    mmap_mode="r"
)


fcgr_index = pd.read_csv(
    FCGR_INDEX_PATH,
    sep="\t",
    dtype={"id": str}
)


id_to_fcgr_row = dict(
    zip(
        fcgr_index["id"],
        fcgr_index["fcgr_row"]
    )
)


missing_train = (
    ~train_metadata["id"]
    .isin(id_to_fcgr_row)
).sum()


missing_val = (
    ~val_metadata["id"]
    .isin(id_to_fcgr_row)
).sum()


print("=" * 72)
print("FCGR")
print("=" * 72)

print(
    "Shape:",
    fcgr_memmap.shape
)

print(
    "Missing train:",
    missing_train
)

print(
    "Missing validation:",
    missing_val
)


assert fcgr_memmap.shape[1:] == (
    IMAGE_SIZE,
    IMAGE_SIZE
)

assert missing_train == 0

assert missing_val == 0

FCGR
Shape: (150272, 64, 64)
Missing train: 0
Missing validation: 0


In [6]:
# ============================================================
# RANDOM PAIR DATASET
# ============================================================

class RandomPairDataset(Dataset):

    def __init__(
        self,
        metadata,
        fcgr_memmap,
        id_to_row,
        n_pairs,
        positive_fraction=0.50,
        seed=42,
        dynamic=True
    ):

        self.metadata = (
            metadata
            .copy()
            .reset_index(drop=True)
        )

        self.fcgr_memmap = fcgr_memmap

        self.n_pairs = int(
            n_pairs
        )

        self.positive_fraction = float(
            positive_fraction
        )

        self.seed = int(
            seed
        )

        self.dynamic = bool(
            dynamic
        )


        self.rows = (
            self.metadata["id"]
            .astype(str)
            .map(id_to_row)
            .to_numpy(dtype=np.int64)
        )


        self.labels = (
            self.metadata["class_id"]
            .to_numpy(dtype=np.int64)
        )


        self.classes = np.array(
            sorted(
                np.unique(
                    self.labels
                )
            ),
            dtype=np.int64
        )


        self.class_to_indices = {

            int(c):
                np.where(
                    self.labels == c
                )[0]

            for c in self.classes
        }


        self._generate_pairs(
            self.seed
        )


    def _generate_pairs(
        self,
        seed
    ):

        rng = np.random.default_rng(
            seed
        )


        n_positive = int(
            round(
                self.n_pairs
                *
                self.positive_fraction
            )
        )


        targets = np.zeros(
            self.n_pairs,
            dtype=np.float32
        )

        targets[
            :n_positive
        ] = 1.0


        rng.shuffle(
            targets
        )


        idx1 = rng.integers(
            0,
            len(self.labels),
            size=self.n_pairs
        )


        idx2 = np.empty(
            self.n_pairs,
            dtype=np.int64
        )


        for i in range(
            self.n_pairs
        ):

            anchor_idx = int(
                idx1[i]
            )

            anchor_class = int(
                self.labels[
                    anchor_idx
                ]
            )


            if targets[i] == 1.0:

                candidates = (
                    self.class_to_indices[
                        anchor_class
                    ]
                )

                partner_idx = anchor_idx


                while (
                    partner_idx
                    ==
                    anchor_idx
                ):

                    partner_idx = int(
                        rng.choice(
                            candidates
                        )
                    )


            else:

                negative_classes = (
                    self.classes[
                        self.classes
                        !=
                        anchor_class
                    ]
                )


                negative_class = int(
                    rng.choice(
                        negative_classes
                    )
                )


                partner_idx = int(
                    rng.choice(
                        self.class_to_indices[
                            negative_class
                        ]
                    )
                )


            idx2[i] = (
                partner_idx
            )


        self.row1 = (
            self.rows[idx1]
        )

        self.row2 = (
            self.rows[idx2]
        )

        self.targets = targets


    def set_epoch(
        self,
        epoch
    ):

        if self.dynamic:

            self._generate_pairs(
                self.seed
                +
                int(epoch)
                *
                100_003
            )


    def __len__(
        self
    ):

        return self.n_pairs


    def __getitem__(
        self,
        index
    ):

        x1 = np.array(
            self.fcgr_memmap[
                int(self.row1[index])
            ],
            dtype=np.float32,
            copy=True
        )


        x2 = np.array(
            self.fcgr_memmap[
                int(self.row2[index])
            ],
            dtype=np.float32,
            copy=True
        )


        return {

            "x1":
                torch.from_numpy(
                    x1
                ).unsqueeze(0),

            "x2":
                torch.from_numpy(
                    x2
                ).unsqueeze(0),

            "target":
                torch.tensor(
                    self.targets[index],
                    dtype=torch.float32
                )
        }

In [7]:
# ============================================================
# LOADERS
# ============================================================

train_pair_dataset = RandomPairDataset(
    metadata=train_metadata,
    fcgr_memmap=fcgr_memmap,
    id_to_row=id_to_fcgr_row,
    n_pairs=TRAIN_PAIRS_PER_EPOCH,
    positive_fraction=POSITIVE_FRACTION,
    seed=RANDOM_STATE,
    dynamic=True
)


val_pair_dataset = RandomPairDataset(
    metadata=val_metadata,
    fcgr_memmap=fcgr_memmap,
    id_to_row=id_to_fcgr_row,
    n_pairs=VAL_PAIRS,
    positive_fraction=POSITIVE_FRACTION,
    seed=RANDOM_STATE + 50_000,
    dynamic=False
)


train_pair_loader = DataLoader(
    train_pair_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=AMP_ENABLED
)


val_pair_loader = DataLoader(
    val_pair_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=AMP_ENABLED
)


print(
    "Train pairs:",
    len(train_pair_dataset)
)

print(
    "Positive fraction:",
    train_pair_dataset.targets.mean()
)

print(
    "Validation pairs:",
    len(val_pair_dataset)
)

Train pairs: 50000
Positive fraction: 0.5
Validation pairs: 10000


In [13]:
# ============================================================
# VISION TRANSFORMER ENCODER — PATCH-NORM VERSION
# ============================================================

class FCGRVisionTransformer(nn.Module):

    def __init__(
        self,
        image_size=64,
        patch_size=8,
        dim=128,
        n_heads=4,
        n_layers=4,
        mlp_dim=512,
        embedding_dim=128,
        dropout=0.10
    ):

        super().__init__()

        assert image_size % patch_size == 0

        self.image_size = image_size
        self.patch_size = patch_size

        patches_per_side = (
            image_size // patch_size
        )

        self.n_patches = (
            patches_per_side ** 2
        )

        # 1 channel × 8 × 8
        patch_dim = (
            patch_size
            *
            patch_size
        )


        # ====================================================
        # PATCHIFY
        # ====================================================

        self.unfold = nn.Unfold(
            kernel_size=patch_size,
            stride=patch_size
        )


        # ====================================================
        # PATCH EMBEDDING
        # ====================================================

        self.patch_embedding = nn.Sequential(

            nn.LayerNorm(
                patch_dim
            ),

            nn.Linear(
                patch_dim,
                dim
            ),

            nn.LayerNorm(
                dim
            )
        )


        # ====================================================
        # CLS TOKEN + POSITIONAL EMBEDDING
        # ====================================================

        self.cls_token = nn.Parameter(
            torch.zeros(
                1,
                1,
                dim
            )
        )


        self.pos_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.n_patches + 1,
                dim
            )
        )


        self.embedding_dropout = (
            nn.Dropout(dropout)
        )


        # ====================================================
        # TRANSFORMER
        # ====================================================

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=dim,
                nhead=n_heads,
                dim_feedforward=mlp_dim,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True
            )
        )


        self.transformer = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=n_layers
            )
        )


        self.final_norm = (
            nn.LayerNorm(dim)
        )


        # ====================================================
        # EMBEDDING HEAD
        # ====================================================

        self.embedding_head = nn.Sequential(

            nn.Linear(
                dim,
                256
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                256,
                embedding_dim
            )
        )


        # ====================================================
        # INITIALIZATION
        # ====================================================

        nn.init.trunc_normal_(
            self.cls_token,
            std=0.02
        )

        nn.init.trunc_normal_(
            self.pos_embedding,
            std=0.02
        )


    def forward(
        self,
        x
    ):

        # ----------------------------------------------------
        # PATCHIFY
        #
        # [B,1,64,64]
        # ->
        # [B,64,64]
        #
        # dimensioni:
        # 64 patch
        # ciascuna da 8×8 = 64 valori
        # ----------------------------------------------------

        x = self.unfold(x)

        x = x.transpose(
            1,
            2
        )


        # ----------------------------------------------------
        # PATCH NORMALIZATION + PROJECTION
        #
        # [B,64,64]
        # ->
        # [B,64,128]
        # ----------------------------------------------------

        x = self.patch_embedding(
            x
        )


        batch_size = (
            x.shape[0]
        )


        # ----------------------------------------------------
        # CLS TOKEN
        # ----------------------------------------------------

        cls = self.cls_token.expand(
            batch_size,
            -1,
            -1
        )


        x = torch.cat(
            [cls, x],
            dim=1
        )


        # ----------------------------------------------------
        # POSITION
        # ----------------------------------------------------

        x = (
            x
            +
            self.pos_embedding
        )


        x = self.embedding_dropout(
            x
        )


        # ----------------------------------------------------
        # TRANSFORMER
        # ----------------------------------------------------

        x = self.transformer(
            x
        )

        x = self.final_norm(
            x
        )


        # ----------------------------------------------------
        # CLS REPRESENTATION
        # ----------------------------------------------------

        cls_output = (
            x[:, 0]
        )


        # ----------------------------------------------------
        # FINAL EMBEDDING
        # ----------------------------------------------------

        z = self.embedding_head(
            cls_output
        )


        return F.normalize(
            z,
            p=2,
            dim=1,
            eps=1e-8
        )

In [14]:
# ============================================================
# SIAMESE ViT
# ============================================================

class SiameseViT(nn.Module):

    def __init__(
        self,
        embedding_dim=128
    ):

        super().__init__()


        self.encoder = FCGRVisionTransformer(
            image_size=IMAGE_SIZE,
            patch_size=PATCH_SIZE,
            dim=VIT_DIM,
            n_heads=VIT_HEADS,
            n_layers=VIT_LAYERS,
            mlp_dim=VIT_MLP_DIM,
            embedding_dim=embedding_dim,
            dropout=VIT_DROPOUT
        )


    def forward(
        self,
        x1,
        x2
    ):

        batch_size = (
            x1.shape[0]
        )


        # Un unico forward del medesimo encoder
        x = torch.cat(
            [x1, x2],
            dim=0
        )


        z = self.encoder(
            x
        )


        return (
            z[:batch_size],
            z[batch_size:]
        )


model = SiameseViT(
    embedding_dim=EMBEDDING_DIM
).to(DEVICE)


n_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print("=" * 72)
print("SIAMESE ViT")
print("=" * 72)

print(
    "Trainable parameters:",
    f"{n_parameters:,}"
)

print(
    "Tokens:",
    model.encoder.n_patches + 1
)

SIAMESE ViT
Trainable parameters: 876,416
Tokens: 65


C:\Users\simon\AppData\Local\Temp\ipykernel_1904\2806918834.py:118: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  nn.TransformerEncoder(


In [15]:
# ============================================================
# RECREATE PATCH-NORM SIAMESE ViT
# ============================================================

set_seed(
    RANDOM_STATE
)


model = SiameseViT(
    embedding_dim=EMBEDDING_DIM
).to(DEVICE)


n_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print("=" * 72)
print("SIAMESE ViT — PATCH NORM")
print("=" * 72)

print(
    "Trainable parameters:",
    f"{n_parameters:,}"
)

print(
    "Tokens:",
    model.encoder.n_patches + 1
)

SIAMESE ViT — PATCH NORM
Trainable parameters: 876,416
Tokens: 65


C:\Users\simon\AppData\Local\Temp\ipykernel_1904\2806918834.py:118: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  nn.TransformerEncoder(


In [10]:
# ============================================================
# EUCLIDEAN CONTRASTIVE LOSS
# ============================================================

class EuclideanContrastiveLoss(nn.Module):

    def __init__(
        self,
        margin=1.25
    ):

        super().__init__()

        self.margin = float(
            margin
        )


    def forward(
        self,
        z1,
        z2,
        target
    ):

        distances = torch.linalg.vector_norm(
            z1.float()
            -
            z2.float(),
            ord=2,
            dim=1
        )


        positive_loss = (
            target
            *
            distances.pow(2)
        )


        negative_loss = (
            (1.0 - target)
            *
            F.relu(
                self.margin
                -
                distances
            ).pow(2)
        )


        loss = (
            positive_loss
            +
            negative_loss
        ).mean()


        return (
            loss,
            distances
        )


criterion = EuclideanContrastiveLoss(
    margin=EUCLIDEAN_MARGIN
)

In [16]:
# ============================================================
# SANITY CHECK
# ============================================================

batch = next(
    iter(train_pair_loader)
)


x1 = batch["x1"].to(
    DEVICE
)

x2 = batch["x2"].to(
    DEVICE
)

target = batch["target"].to(
    DEVICE
)


model.eval()


with torch.no_grad():

    z1, z2 = model(
        x1,
        x2
    )


    loss, distances = criterion(
        z1,
        z2,
        target
    )


print("=" * 72)
print("ViT SANITY CHECK")
print("=" * 72)

print(
    "Input:",
    x1.shape
)

print(
    "Embedding 1:",
    z1.shape
)

print(
    "Embedding 2:",
    z2.shape
)

print(
    "Mean norm:",
    z1.norm(
        dim=1
    ).mean().item()
)

print(
    "Loss:",
    loss.item()
)

print(
    "Mean distance:",
    distances.mean().item()
)

print(
    "Finite:",
    torch.isfinite(
        loss
    ).item()
)

ViT SANITY CHECK
Input: torch.Size([64, 1, 64, 64])
Embedding 1: torch.Size([64, 128])
Embedding 2: torch.Size([64, 128])
Mean norm: 1.0
Loss: 0.42363858222961426
Mean distance: 0.3702954053878784
Finite: True


In [12]:
# ============================================================
# DIAGNOSTIC — RAW FCGR + PATCH EMBEDDING VARIABILITY
# ============================================================

model.eval()

with torch.no_grad():

    raw = x1.float()

    patch = model.encoder.patch_embedding(raw)

    patch_tokens = (
        patch
        .flatten(2)
        .transpose(1, 2)
    )


print("=" * 72)
print("ViT INPUT DIAGNOSTIC")
print("=" * 72)

print("RAW FCGR")
print("min :", raw.min().item())
print("max :", raw.max().item())
print("mean:", raw.mean().item())
print("std :", raw.std().item())

print()

print("PATCH EMBEDDING")
print("shape:", patch_tokens.shape)
print("mean :", patch_tokens.mean().item())
print("std  :", patch_tokens.std().item())

# variabilità tra campioni per lo stesso token
sample_variability = (
    patch_tokens
    .std(dim=0)
    .mean()
    .item()
)

print(
    "Mean variability across samples:",
    sample_variability
)

ViT INPUT DIAGNOSTIC
RAW FCGR
min : 0.0
max : 0.17449665069580078
mean: 0.000244140625
std : 0.0010943158995360136

PATCH EMBEDDING
shape: torch.Size([64, 64, 128])
mean : 0.004965700209140778
std  : 0.0744594857096672
Mean variability across samples: 0.0005541737773455679


In [17]:
# ============================================================
# TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader,
    dataset,
    criterion,
    optimizer,
    scaler,
    epoch
):

    model.train()

    dataset.set_epoch(epoch)

    total_loss = 0.0
    total_samples = 0

    all_targets = []
    all_distances = []

    start_time = time.perf_counter()


    for batch in loader:

        x1 = batch["x1"].to(
            DEVICE,
            non_blocking=True
        )

        x2 = batch["x2"].to(
            DEVICE,
            non_blocking=True
        )

        target = batch["target"].to(
            DEVICE,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=AMP_ENABLED
        ):

            z1, z2 = model(
                x1,
                x2
            )


        loss, distances = criterion(
            z1,
            z2,
            target
        )


        if AMP_ENABLED:

            scaler.scale(
                loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            scaler.step(
                optimizer
            )

            scaler.update()

        else:

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()


        n = target.shape[0]

        total_loss += (
            loss.detach().item()
            * n
        )

        total_samples += n


        all_targets.append(
            target.detach()
            .cpu()
            .numpy()
        )

        all_distances.append(
            distances.detach()
            .cpu()
            .numpy()
        )


    targets = np.concatenate(
        all_targets
    )

    distances = np.concatenate(
        all_distances
    )


    auc = roc_auc_score(
        targets,
        -distances
    )


    return {
        "loss":
            total_loss
            /
            total_samples,

        "auc":
            float(auc),

        "seconds":
            time.perf_counter()
            -
            start_time
    }

In [18]:
# ============================================================
# VALIDATION
# ============================================================

@torch.no_grad()
def evaluate_pairwise(
    model,
    loader,
    criterion
):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_targets = []
    all_distances = []


    for batch in loader:

        x1 = batch["x1"].to(
            DEVICE,
            non_blocking=True
        )

        x2 = batch["x2"].to(
            DEVICE,
            non_blocking=True
        )

        target = batch["target"].to(
            DEVICE,
            non_blocking=True
        )


        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=AMP_ENABLED
        ):

            z1, z2 = model(
                x1,
                x2
            )


        loss, distances = criterion(
            z1,
            z2,
            target
        )


        n = target.shape[0]

        total_loss += (
            loss.item()
            * n
        )

        total_samples += n


        all_targets.append(
            target.cpu()
            .numpy()
        )

        all_distances.append(
            distances.cpu()
            .numpy()
        )


    targets = np.concatenate(
        all_targets
    )

    distances = np.concatenate(
        all_distances
    )


    positive = distances[
        targets == 1
    ]

    negative = distances[
        targets == 0
    ]


    d_pos = float(
        positive.mean()
    )

    d_neg = float(
        negative.mean()
    )


    gap = (
        d_neg
        -
        d_pos
    )


    pooled_variance = (
        0.5
        *
        (
            positive.var()
            +
            negative.var()
        )
    )


    d_prime = float(
        gap
        /
        np.sqrt(
            pooled_variance
            +
            1e-12
        )
    )


    auc = roc_auc_score(
        targets,
        -distances
    )


    return {
        "loss":
            total_loss
            /
            total_samples,

        "auc":
            float(auc),

        "d_pos":
            d_pos,

        "d_neg":
            d_neg,

        "gap":
            gap,

        "d_prime":
            d_prime
    }

In [19]:
# ============================================================
# ViT BENCHMARK
# ============================================================

benchmark_model = SiameseViT(
    embedding_dim=EMBEDDING_DIM
).to(DEVICE)

benchmark_optimizer = torch.optim.AdamW(
    benchmark_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

benchmark_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)


benchmark_model.train()


batch = next(
    iter(train_pair_loader)
)


x1 = batch["x1"].to(DEVICE)
x2 = batch["x2"].to(DEVICE)
target = batch["target"].to(DEVICE)


if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()


# warm-up
for _ in range(3):

    benchmark_optimizer.zero_grad(
        set_to_none=True
    )

    with torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16,
        enabled=AMP_ENABLED
    ):

        z1, z2 = benchmark_model(
            x1,
            x2
        )

        loss, _ = criterion(
            z1,
            z2,
            target
        )

    benchmark_scaler.scale(
        loss
    ).backward()

    benchmark_scaler.step(
        benchmark_optimizer
    )

    benchmark_scaler.update()


if DEVICE.type == "cuda":
    torch.cuda.synchronize()


N_BENCHMARK = 10

start = time.perf_counter()


for _ in range(N_BENCHMARK):

    benchmark_optimizer.zero_grad(
        set_to_none=True
    )

    with torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16,
        enabled=AMP_ENABLED
    ):

        z1, z2 = benchmark_model(
            x1,
            x2
        )

        loss, _ = criterion(
            z1,
            z2,
            target
        )

    benchmark_scaler.scale(
        loss
    ).backward()

    benchmark_scaler.step(
        benchmark_optimizer
    )

    benchmark_scaler.update()


if DEVICE.type == "cuda":
    torch.cuda.synchronize()


seconds_per_batch = (
    time.perf_counter()
    -
    start
) / N_BENCHMARK


batches_per_epoch = len(
    train_pair_loader
)


estimated_epoch = (
    seconds_per_batch
    *
    batches_per_epoch
)


print("=" * 72)
print("ViT BENCHMARK")
print("=" * 72)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Batches/epoch:",
    batches_per_epoch
)

print(
    "Seconds/batch:",
    f"{seconds_per_batch:.4f}"
)

print(
    "Estimated epoch:",
    f"{estimated_epoch:.1f}s"
)

print(
    "Estimated epoch:",
    f"{estimated_epoch / 60:.2f} min"
)

if DEVICE.type == "cuda":

    peak_memory = (
        torch.cuda.max_memory_allocated()
        /
        1024**3
    )

    print(
        "Peak GPU memory:",
        f"{peak_memory:.2f} GB"
    )

C:\Users\simon\AppData\Local\Temp\ipykernel_1904\2806918834.py:118: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  nn.TransformerEncoder(


ViT BENCHMARK
Batch size: 64
Batches/epoch: 782
Seconds/batch: 0.0187
Estimated epoch: 14.6s
Estimated epoch: 0.24 min
Peak GPU memory: 0.25 GB


In [20]:
# ============================================================
# ViT SMOKE TEST
# ============================================================

SMOKE_EPOCHS = 3


set_seed(
    RANDOM_STATE
)


model = SiameseViT(
    embedding_dim=EMBEDDING_DIM
).to(DEVICE)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=AMP_ENABLED
)


print("=" * 120)
print("SIAMESE ViT PATCH-NORM k=6 — SMOKE TEST")
print("=" * 120)


for epoch in range(
    1,
    SMOKE_EPOCHS + 1
):

    train_metrics = train_one_epoch(
        model=model,
        loader=train_pair_loader,
        dataset=train_pair_dataset,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        epoch=epoch
    )


    val_metrics = evaluate_pairwise(
        model=model,
        loader=val_pair_loader,
        criterion=criterion
    )


    print(
        f"Epoch {epoch:02d}/{SMOKE_EPOCHS}"

        f" | train loss "
        f"{train_metrics['loss']:.4f}"

        f" | train AUC "
        f"{train_metrics['auc']:.4f}"

        f" | val loss "
        f"{val_metrics['loss']:.4f}"

        f" | val AUC "
        f"{val_metrics['auc']:.4f}"

        f" | d+ "
        f"{val_metrics['d_pos']:.4f}"

        f" | d- "
        f"{val_metrics['d_neg']:.4f}"

        f" | gap "
        f"{val_metrics['gap']:.4f}"

        f" | d' "
        f"{val_metrics['d_prime']:.4f}"

        f" | "
        f"{train_metrics['seconds']:.1f}s"
    )

C:\Users\simon\AppData\Local\Temp\ipykernel_1904\2806918834.py:118: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  nn.TransformerEncoder(


SIAMESE ViT PATCH-NORM k=6 — SMOKE TEST
Epoch 01/3 | train loss 0.3787 | train AUC 0.6093 | val loss 0.4084 | val AUC 0.6119 | d+ 0.4929 | d- 0.6036 | gap 0.1107 | d' 0.3964 | 17.1s
Epoch 02/3 | train loss 0.3707 | train AUC 0.6311 | val loss 0.3950 | val AUC 0.6284 | d+ 0.5370 | d- 0.6685 | gap 0.1315 | d' 0.4599 | 16.8s
Epoch 03/3 | train loss 0.3684 | train AUC 0.6391 | val loss 0.3912 | val AUC 0.6238 | d+ 0.4859 | d- 0.5893 | gap 0.1034 | d' 0.4415 | 17.4s
